# 第12周｜冻结、测试与复现报告

学习目标：分析方案冻结、测试集一次性评价、可复现成果。

__数据：默认使用官方 PRIME-CVD 5 万人数据__（`python run.py download` 下载，来源与 MD5 核验记录在 `data/raw/manifest.json`）。把 `MODE` 改成 `"demo"` 可以切换到离线演示数据，但演示数据是另外生成的，**不是 PRIME-CVD**，结果不能当作官方结果。

本周验收：从源文件重新生成主要结果；自己解释关键结果及局限。

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# 从项目根目录、notebooks目录或其他项目内目录启动均可。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists()), None)
if ROOT is None:
    raise RuntimeError("请在解压后的项目目录中启动 Jupyter。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from primecvd.common import load_config, provenance_label
from primecvd.pipeline import prepare
from primecvd.splitting import select_split

MODE = "official"  # 官方数据；离线练习或跑测试时可改为 "demo"（演示数据不是 PRIME-CVD）
cfg = load_config(ROOT / "configs" / f"{MODE}.json")
cfg["output_dir"] = f"outputs/notebooks_{MODE}"
print(provenance_label(cfg))
df, splits = prepare(cfg)
train = select_split(df, splits, "train")
valid = select_split(df, splits, "valid")
# 不创建test DataFrame；模型试验只使用train和valid。
print("队列、训练集、验证集：", len(df), len(train), len(valid))

# 命令行绘图模块采用无窗口后端；在Notebook中显式启用行内图像。
get_ipython().run_line_magic("matplotlib", "inline")
from primecvd import pubplot as pp   # 与精讲篇相同的顶刊作图风格（在启用行内图像之后设置）
pp.setup()

OFFICIAL SOURCE FILES / 官方下载文件
队列、训练集、验证集： 50000 35000 7500


## 完整项目：运行、冻结、报告
本Notebook使用独立输出目录，避免覆盖每周练习文件。它会运行训练内CV、验证评价与EMR重建，生成自包含HTML报告，但**不会自动打开最终测试集**。

In [2]:
from primecvd.pipeline import run_all
cfg["output_dir"] = f"outputs/notebook_capstone_{MODE}"
report_path = run_all(cfg)
print("完整报告：", report_path)
frozen_path = report_path.parent/"frozen_spec.json"
display(json.loads(frozen_path.read_text(encoding="utf-8")))

  CV alpha=0, fold=1, Brier=0.02771


  CV alpha=0, fold=2, Brier=0.02826


  CV alpha=0, fold=3, Brier=0.02774


  CV alpha=0.001, fold=1, Brier=0.02773


  CV alpha=0.001, fold=2, Brier=0.02825


...

  CV alpha=0.01, fold=1, Brier=0.02794


  CV alpha=0.01, fold=2, Brier=0.02831


  CV alpha=0.01, fold=3, Brier=0.02772


完整报告： outputs/notebook_capstone_official/report.html


{'config_sha256': '8a553821a6e49115eca0074109f28ca3bf2b8ae97886508dd002978fcf7b6dec',
 'data_sha256': '34ae149df24e8dfa66371def307bb6784652d8494b89c6d2bfe82cfa4184cdf8',
 'splits_sha256': '469def40aa634f3ba9f71cc866538e5057b733618a0a78c3ee72c63778885fae',
 'model_sha256': 'c4e3eec4fbb2828f11b7a83f5e2bafe6cc75730e61766873a8b4d5c2f0f1d3ff',
 'calibration_sha256': 'aba4804b4d7d8456c5c217f6bc0f2a488188a06cc656e3a1c855309f889420f5',
 'selected_alpha': 0.001,
 'selection_rule': 'Minimum mean training-only CV 5-year IPCW Brier; validation not used for model selection.',
 'final_fit_set': 'train only (70%); no train+valid refit',
 'test_opened': False,
 'source': 'OFFICIAL SOURCE FILES / 官方下载文件'}

## 先写方法，再打开测试集
确认变量、时间窗、处理规则和选择方式都已冻结。在数据是demo时，这只是流程练习，不得把得到的结果当作官方PRIME-CVD结果或临床证据。对真实的新官方文件，测试患者在此前不应被反复检查或用于模型选择。

In [3]:
# 第12周完成方案和方法记录后，手动取消下面两行注释：
# from primecvd.pipeline import finalize
# finalize(cfg, unlock_test=True)
# 测试打开后，本输出目录将阻止重训；不要通过改目录逃避这个约束。

## 最终交付练习
按照 `docs/08_final_report_template.md` 写3–5页自己的报告，并附复现命令。说明数据来源、患者划分、普通/惩罚Cox、删失假设、校准、清洗规则及局限。每个数值都来自实际输出；不能把“尚未运行”写成“结果良好”。

命令行 `python run.py all` 生成的官方结果在 `outputs/official/report.html`，可以和本笔记本生成的报告对照。不必让自己的所有试验都与参考结果相同；必须能解释差异，并保留配置与来源记录。

---
本周复盘：请用自己的话记录一个学会的概念、一个发现的错误和一个待解决问题。

特别提醒：测试后不反复挑模型；若使用 demo 数据，报告必须写明"演示数据，不是 PRIME-CVD"。